# Comparing Two TEI Editions of Aristotle's *Poetics*

This notebook compares:
- **Butcher 1911** (`tlg0086_tlg034_butcher1911-grc.xml`) — LLM-produced TEI with `<lb>` line breaks and end-of-line hyphens
- **Perseus/Kassel** (`tlg0086_tlg034_perseus-grc2.xml`) — the reference Perseus Digital Library edition

Goals:
1. Verify the chapter/section div structure matches
2. Verify **no words were dropped** in the Butcher edition
3. Surface genuine textual variants (different editorial readings) vs. processing errors

In [113]:
import re, unicodedata, json
from collections import OrderedDict, defaultdict
from lxml import etree
from difflib import SequenceMatcher, unified_diff
from IPython.display import display, HTML, Markdown

## 1. Parse both XML files

In [114]:
NS = {'tei': 'http://www.tei-c.org/ns/1.0'}

butcher_tree = etree.parse('/Users/gcrane/github/Poetics2.0/grc/tlg0086.tlg034.butcher1911-grc.xml')
perseus_tree = etree.parse('/Users/gcrane/github/canonical-greekLit/data/tlg0086/tlg034/tlg0086.tlg034.perseus-grc2.xml')

print('Butcher root tag:', butcher_tree.getroot().tag)
print('Perseus root tag:', perseus_tree.getroot().tag)

Butcher root tag: {http://www.tei-c.org/ns/1.0}TEI
Perseus root tag: {http://www.tei-c.org/ns/1.0}TEI


## 2. Extract text from each section

The key challenge: Butcher's text has `<lb>` line breaks and **end-of-line hyphens** where a word is split across two lines (e.g. `πρῶ-` / `τον`). We need to rejoin those before comparing.

Both files also have `<del>`, `<add>`, `<quote>`, `[cite: N]` markers, and `<note>` elements that must be handled.

In [115]:
def extract_text_from_element(el):
    """Recursively extract all text content from an XML element,
    including text inside <add>, <del>, <quote> children,
    but EXCLUDING <note> elements.
    Treats <lb/> as a space boundary.
    """
    parts = []
    
    def _walk(node):
        tag = etree.QName(node.tag).localname if isinstance(node.tag, str) else ''
        
        # Skip <note> elements entirely
        if tag == 'note':
            if node.tail:
                parts.append(node.tail)
            return
        
        # For <lb>, <pb>, <milestone> — no text content, but insert a space
        # to ensure words on different lines don't merge
        if tag in ('lb', 'pb', 'milestone'):
            parts.append(' ')
            if node.tail:
                parts.append(node.tail)
            return
        
        # Normal element: take its .text, recurse children, then .tail
        if node.text:
            parts.append(node.text)
        for child in node:
            _walk(child)
        if node.tail:
            parts.append(node.tail)
    
    _walk(el)
    return ''.join(parts)


def normalize_greek(text):
    """Normalize extracted Greek text for comparison:
    - rejoin hyphenated words (πρῶ- τον → πρῶτον)
    - strip [cite: N] markers
    - normalize unicode (NFC)
    - collapse whitespace
    - strip editorial brackets [] <> from the text content
    - normalize various apostrophe/breathing characters
    """
    # Remove [cite: N] markers
    text = re.sub(r'\[cite:\s*\d+\]', '', text)
    
    # Rejoin hyphenated line-break words: word- \s* nextword → wordnextword
    # The hyphen at end of a word fragment followed by whitespace (possibly with
    # newlines) and then the continuation
    text = re.sub(r'(\S)-\s+', r'\1', text)
    
    # NFC normalize
    text = unicodedata.normalize('NFC', text)
    
    # Normalize various apostrophes and right single quotes to a standard one
    text = text.replace('\u2019', '\u0027')  # ' → '
    text = text.replace('\u02BC', '\u0027')  # ʼ → '
    text = text.replace('\u1FBD', '\u0027')  # ᾽ → '
    text = text.replace('\u2018', '')         # left single quote, remove
    
    # Strip . . . ellipsis markers (Perseus uses these for lacunae)
    text = re.sub(r'\.\s*\.\s*\.\s*\.?', '', text)
    
    # Collapse whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text


# Quick test with a Butcher fragment
test = "πρῶ- \n            τον ἀπὸ τῶν πρώτων. "
print(repr(normalize_greek(test)))
# Should be: 'πρῶτον ἀπὸ τῶν πρώτων.'

def section_sort_key(s):
    """Sort key for section numbers that may have letter suffixes (e.g. '11b', '10c')."""
    m = re.match(r'(\d+)(.*)', s)
    if m:
        return (int(m.group(1)), m.group(2))
    return (0, s)


'πρῶτον ἀπὸ τῶν πρώτων.'


## 3. Build chapter → section dictionaries

In [116]:
def build_chapter_dict_butcher(tree):
    """Returns {chapter_n: {section_n: normalized_text, ...}, ...}
    
    Note: Butcher has duplicate chapter divs (e.g. two <div type='chapter' n='4'>)
    that represent a page-break split. We merge them.
    """
    chapters = OrderedDict()
    for ch_div in tree.xpath('//tei:div[@type="chapter"]', namespaces=NS):
        ch_n = ch_div.get('n')
        if ch_n not in chapters:
            chapters[ch_n] = OrderedDict()
        for sec_div in ch_div.xpath('tei:div[@type="section"]', namespaces=NS):
            sec_n = sec_div.get('n')
            raw = extract_text_from_element(sec_div)
            chapters[ch_n][sec_n] = normalize_greek(raw)
    return chapters


def build_chapter_dict_perseus(tree):
    """Returns {chapter_n: {section_n: normalized_text, ...}, ...}"""
    chapters = OrderedDict()
    for ch_div in tree.xpath('//tei:div[@subtype="chapter"]', namespaces=NS):
        ch_n = ch_div.get('n')
        chapters[ch_n] = OrderedDict()
        for sec_div in ch_div.xpath('tei:div[@subtype="subchapter"]', namespaces=NS):
            sec_n = sec_div.get('n')
            raw = extract_text_from_element(sec_div)
            chapters[ch_n][sec_n] = normalize_greek(raw)
    return chapters


butcher = build_chapter_dict_butcher(butcher_tree)
perseus = build_chapter_dict_perseus(perseus_tree)

print(f'Butcher: {len(butcher)} chapters')
print(f'Perseus: {len(perseus)} chapters')

Butcher: 26 chapters
Perseus: 26 chapters


## 4. Compare chapter/section structure

In [117]:
all_chapters = sorted(set(butcher.keys()) | set(perseus.keys()), key=int)

rows = []
for ch in all_chapters:
    b_secs = sorted(butcher.get(ch, {}).keys(), key=section_sort_key) if ch in butcher else []
    p_secs = sorted(perseus.get(ch, {}).keys(), key=section_sort_key) if ch in perseus else []
    b_set = set(b_secs)
    p_set = set(p_secs)
    only_b = sorted(b_set - p_set, key=section_sort_key) if b_set - p_set else []
    only_p = sorted(p_set - b_set, key=section_sort_key) if p_set - b_set else []
    match = '✅' if b_set == p_set else '❌'
    rows.append({
        'chapter': ch,
        'butcher_sections': len(b_secs),
        'perseus_sections': len(p_secs),
        'match': match,
        'only_in_butcher': ', '.join(only_b) if only_b else '',
        'only_in_perseus': ', '.join(only_p) if only_p else '',
    })

# Display as HTML table
html = '<table border="1" cellpadding="4" style="border-collapse:collapse">'
html += '<tr><th>Chapter</th><th>Butcher §</th><th>Perseus §</th><th>Match</th><th>Only Butcher</th><th>Only Perseus</th></tr>'
for r in rows:
    color = '' if r['match'] == '✅' else ' style="background:#ffe0e0"'
    html += f'<tr{color}>'
    html += f'<td>{r["chapter"]}</td><td>{r["butcher_sections"]}</td><td>{r["perseus_sections"]}</td>'
    html += f'<td>{r["match"]}</td><td>{r["only_in_butcher"]}</td><td>{r["only_in_perseus"]}</td>'
    html += '</tr>'
html += '</table>'
display(HTML(html))


Chapter,Butcher §,Perseus §,Match,Only Butcher,Only Perseus
1,14,14,✅,,
2,7,7,✅,,
3,7,7,✅,,
4,21,21,✅,,
5,11,11,✅,,
6,28,28,✅,,
7,12,12,✅,,
8,4,4,✅,,
9,15,15,✅,,
10,4,4,✅,,


## 5. Word-level comparison per chapter

For each chapter, we concatenate all section text (in section order) and tokenize into words. Then we use `SequenceMatcher` to find **missing** and **extra** words.

This catches dropped words regardless of section boundary differences.

In [118]:
def tokenize(text):
    """Split Greek text into word tokens, stripping punctuation."""
    # Keep Greek letters, combining marks, apostrophes
    # Remove other punctuation but preserve word boundaries
    tokens = text.split()
    # Strip leading/trailing punctuation from each token
    cleaned = []
    for t in tokens:
        t = re.sub(r'^[^\w\u0370-\u03FF\u1F00-\u1FFF]+', '', t)
        t = re.sub(r'[^\w\u0370-\u03FF\u1F00-\u1FFF]+$', '', t)
        if t:
            cleaned.append(t)
    return cleaned


def chapter_full_text(ch_dict):
    """Concatenate all sections of a chapter in order."""
    parts = []
    for sec_n in sorted(ch_dict.keys(), key=section_sort_key):
        parts.append(ch_dict[sec_n])
    return ' '.join(parts)


# Quick sanity check
print('Butcher ch1 word count:', len(tokenize(chapter_full_text(butcher['1']))))
print('Perseus ch1 word count:', len(tokenize(chapter_full_text(perseus['1']))))

Butcher ch1 word count: 407
Perseus ch1 word count: 406


In [119]:
def compare_word_lists(words_a, words_b):
    """Compare two word lists using SequenceMatcher.
    Returns (only_in_a, only_in_b, common_count, ratio) where
    only_in_a / only_in_b are lists of (position, word) tuples.
    """
    sm = SequenceMatcher(None, words_a, words_b, autojunk=False)
    only_a = []  # in A but not in B (words Butcher has, Perseus doesn't or vice versa)
    only_b = []
    common = 0
    
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == 'equal':
            common += (i2 - i1)
        elif tag == 'delete':  # in A only
            for k in range(i1, i2):
                only_a.append((k, words_a[k]))
        elif tag == 'insert':  # in B only
            for k in range(j1, j2):
                only_b.append((k, words_b[k]))
        elif tag == 'replace':
            for k in range(i1, i2):
                only_a.append((k, words_a[k]))
            for k in range(j1, j2):
                only_b.append((k, words_b[k]))
    
    ratio = sm.ratio()
    return only_a, only_b, common, ratio

In [120]:
# Run comparison for every chapter
chapter_results = []

for ch in all_chapters:
    b_text = chapter_full_text(butcher.get(ch, {})) if ch in butcher else ''
    p_text = chapter_full_text(perseus.get(ch, {})) if ch in perseus else ''
    
    b_words = tokenize(b_text)
    p_words = tokenize(p_text)
    
    only_b, only_p, common, ratio = compare_word_lists(b_words, p_words)
    
    chapter_results.append({
        'chapter': ch,
        'butcher_words': len(b_words),
        'perseus_words': len(p_words),
        'common': common,
        'only_butcher': only_b,
        'only_perseus': only_p,
        'ratio': ratio,
    })

# Summary table
html = '<h3>Word-level comparison per chapter</h3>'
html += '<table border="1" cellpadding="4" style="border-collapse:collapse; font-size:13px">'
html += '<tr><th>Ch</th><th>Butcher words</th><th>Perseus words</th><th>Shared</th>'
html += '<th>Only Butcher</th><th>Only Perseus</th><th>Similarity</th></tr>'

for r in chapter_results:
    ob = len(r['only_butcher'])
    op = len(r['only_perseus'])
    color = ''
    if ob > 0 or op > 0:
        color = ' style="background:#fff3cd"' if r['ratio'] > 0.95 else ' style="background:#ffe0e0"'
    html += f'<tr{color}>'
    html += f'<td>{r["chapter"]}</td><td>{r["butcher_words"]}</td><td>{r["perseus_words"]}</td>'
    html += f'<td>{r["common"]}</td><td>{ob}</td><td>{op}</td>'
    html += f'<td>{r["ratio"]:.3f}</td></tr>'
html += '</table>'
html += '<p><b>Key:</b> 🟡 Yellow = minor differences (>95% similar), 🔴 Red = larger divergence</p>'
display(HTML(html))

Ch,Butcher words,Perseus words,Shared,Only Butcher,Only Perseus,Similarity
1,407,406,399,8,7,0.982
2,161,161,159,2,2,0.988
3,203,203,201,2,2,0.990
4,570,569,548,22,21,0.962
5,237,236,233,4,3,0.985
6,713,715,688,25,27,0.964
7,318,319,314,4,5,0.986
8,188,188,186,2,2,0.989
9,476,477,468,8,9,0.982
10,90,90,86,4,4,0.956


## 6. Detailed diff — words only in one edition

For each chapter with differences, show the actual words that diverge. This helps distinguish:
- **Dropped words** (processing error) — a word in Perseus but missing from Butcher
- **Genuine variants** — the two editions read differently (e.g. Butcher has `κὰν`, Kassel has `κἀν`)

In [121]:
for r in chapter_results:
    if not r['only_butcher'] and not r['only_perseus']:
        continue
    
    ch = r['chapter']
    print(f"\n{'='*70}")
    print(f"Chapter {ch}  (similarity: {r['ratio']:.3f})")
    print(f"{'='*70}")
    
    if r['only_butcher']:
        print(f"\n  Words ONLY in Butcher ({len(r['only_butcher'])}):\n")
        for pos, w in r['only_butcher']:
            print(f"    [{pos:4d}] {w}")
    
    if r['only_perseus']:
        print(f"\n  Words ONLY in Perseus ({len(r['only_perseus'])}):\n")
        for pos, w in r['only_perseus']:
            print(f"    [{pos:4d}] {w}")


Chapter 1  (similarity: 0.982)

  Words ONLY in Butcher (8):

    [   0] Περὶ
    [ 201] ἢ
    [ 208] μετ᾿
    [ 217] ἀνώνυμος
    [ 218] τυγχάνει
    [ 219] οὖσα
    [ 263] ἐλεγειοποιούς
    [ 339] τοῦτον

  Words ONLY in Perseus (7):

    [   0] περὶ
    [ 201] καὶ
    [ 202] ἡ
    [ 209] μετ
    [ 218] ἀνώνυμοι
    [ 219] τυγχάνουσι
    [ 263] ἐλεγειοποιοὺς

Chapter 2  (similarity: 0.988)

  Words ONLY in Butcher (2):

    [   0] Ἐπεὶ
    [ 139] τῇ

  Words ONLY in Perseus (2):

    [   0] ἐπεὶ
    [ 141] τῇ

Chapter 3  (similarity: 0.990)

  Words ONLY in Butcher (2):

    [   0] Ἔτι
    [  57] κατ᾿

  Words ONLY in Perseus (2):

    [   0] ἔτι
    [  57] κατ

Chapter 4  (similarity: 0.962)

  Words ONLY in Butcher (22):

    [   0] Ἐοίκασι
    [  22] ἐστί
    [ 141] δὴ
    [ 149] και
    [ 160] φανερόν
    [ 164] καὶ
    [ 206] ἅτεροι
    [ 240] ἁρμόττον
    [ 243] μέτρον
    [ 244] διὸ
    [ 282] ὅτι
    [ 289] τὰ
    [ 292] σχήματα
    [ 351] μείζονα
    [ 364] ἄρ
    [ 378] κρ

## 7. Side-by-side section diff (aligned sections only)

For sections that exist in both editions, show a colored inline diff.

In [122]:
def html_word_diff(text_a, text_b, label_a='Butcher', label_b='Perseus'):
    """Produce an HTML fragment highlighting word-level differences."""
    words_a = text_a.split()
    words_b = text_b.split()
    sm = SequenceMatcher(None, words_a, words_b, autojunk=False)
    
    html_a = []
    html_b = []
    
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == 'equal':
            chunk = ' '.join(words_a[i1:i2])
            html_a.append(chunk)
            html_b.append(chunk)
        elif tag == 'delete':
            chunk = ' '.join(words_a[i1:i2])
            html_a.append(f'<span style="background:#ffa0a0;font-weight:bold">{chunk}</span>')
        elif tag == 'insert':
            chunk = ' '.join(words_b[j1:j2])
            html_b.append(f'<span style="background:#a0ffa0;font-weight:bold">{chunk}</span>')
        elif tag == 'replace':
            chunk_a = ' '.join(words_a[i1:i2])
            chunk_b = ' '.join(words_b[j1:j2])
            html_a.append(f'<span style="background:#ffa0a0;font-weight:bold">{chunk_a}</span>')
            html_b.append(f'<span style="background:#a0ffa0;font-weight:bold">{chunk_b}</span>')
    
    return (
        f'<div style="margin:4px 0"><b>{label_a}:</b> {" ".join(html_a)}</div>'
        f'<div style="margin:4px 0"><b>{label_b}:</b> {" ".join(html_b)}</div>'
    )

In [123]:
# Show side-by-side diffs for every section that differs
diff_count = 0
html_parts = ['<h3>Section-level diffs (only sections with differences shown)</h3>']

for ch in all_chapters:
    if ch not in butcher or ch not in perseus:
        continue
    
    shared_secs = sorted(
        set(butcher[ch].keys()) & set(perseus[ch].keys()),
        key=section_sort_key
    )
    
    for sec in shared_secs:
        b_text = butcher[ch][sec]
        p_text = perseus[ch][sec]
        
        if b_text == p_text:
            continue
        
        diff_count += 1
        html_parts.append(
            f'<div style="border:1px solid #ccc; padding:8px; margin:8px 0; border-radius:4px">'
            f'<b>Chapter {ch}, Section {sec}</b><br>'
            f'{html_word_diff(b_text, p_text)}'
            f'</div>'
        )

html_parts.insert(1, f'<p>Total sections with differences: <b>{diff_count}</b></p>')
display(HTML('\n'.join(html_parts)))


## 8. Overall statistics

In [124]:
# Full-text comparison
butcher_full = ' '.join(chapter_full_text(butcher[ch]) for ch in sorted(butcher.keys(), key=section_sort_key))
perseus_full = ' '.join(chapter_full_text(perseus[ch]) for ch in sorted(perseus.keys(), key=section_sort_key))

b_all = tokenize(butcher_full)
p_all = tokenize(perseus_full)

only_b, only_p, common, ratio = compare_word_lists(b_all, p_all)

print(f'Total Butcher words:  {len(b_all)}')
print(f'Total Perseus words:  {len(p_all)}')
print(f'Shared (aligned):     {common}')
print(f'Only in Butcher:      {len(only_b)}')
print(f'Only in Perseus:      {len(only_p)}')
print(f'Overall similarity:   {ratio:.4f}')
print()

if len(only_p) > 0:
    print(f'⚠️  {len(only_p)} words appear in Perseus but NOT in Butcher.')
    print('   These could be dropped words or genuine editorial omissions.')
    print('   Review the per-chapter details above to distinguish.')
else:
    print('✅ No words missing from Butcher relative to Perseus.')

if len(only_b) > 0:
    print(f'ℹ️  {len(only_b)} words appear in Butcher but NOT in Perseus.')
    print('   These could be editorial additions or OCR artifacts.')


Total Butcher words:  10252
Total Perseus words:  10256
Shared (aligned):     9934
Only in Butcher:      318
Only in Perseus:      322
Overall similarity:   0.9688

⚠️  322 words appear in Perseus but NOT in Butcher.
   These could be dropped words or genuine editorial omissions.
   Review the per-chapter details above to distinguish.
ℹ️  318 words appear in Butcher but NOT in Perseus.
   These could be editorial additions or OCR artifacts.


## 9. Classify differences: likely variants vs. likely errors

We try to automatically classify each word-level difference:
- **Orthographic variant**: same word, different accent/breathing/spelling (κὰν vs κἀν)
- **Editorial variant**: different word choice (both legitimate readings)
- **Possible error**: a word present in one but completely absent from the other with no replacement

In [125]:
def strip_accents(s):
    """Remove all combining diacritical marks from Greek text."""
    nfd = unicodedata.normalize('NFD', s)
    return ''.join(c for c in nfd if unicodedata.category(c) not in ('Mn',))


def classify_opcodes(words_a, words_b):
    """Classify each difference between two word lists."""
    sm = SequenceMatcher(None, words_a, words_b, autojunk=False)
    results = []
    
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == 'equal':
            continue
        
        a_chunk = words_a[i1:i2]
        b_chunk = words_b[j1:j2]
        
        if tag == 'replace' and len(a_chunk) == len(b_chunk):
            # One-to-one replacements — check if orthographic
            for wa, wb in zip(a_chunk, b_chunk):
                if strip_accents(wa) == strip_accents(wb):
                    kind = 'orthographic'
                else:
                    kind = 'variant'
                results.append({
                    'type': kind,
                    'butcher': wa,
                    'perseus': wb,
                    'pos_a': i1, 'pos_b': j1,
                })
        elif tag == 'replace':
            results.append({
                'type': 'variant',
                'butcher': ' '.join(a_chunk),
                'perseus': ' '.join(b_chunk),
                'pos_a': i1, 'pos_b': j1,
            })
        elif tag == 'delete':
            results.append({
                'type': 'only_butcher',
                'butcher': ' '.join(a_chunk),
                'perseus': '',
                'pos_a': i1, 'pos_b': j1,
            })
        elif tag == 'insert':
            results.append({
                'type': 'only_perseus',
                'butcher': '',
                'perseus': ' '.join(b_chunk),
                'pos_a': i1, 'pos_b': j1,
            })
    
    return results


# Classify all diffs
all_diffs = classify_opcodes(b_all, p_all)

# Summarize
counts = defaultdict(int)
for d in all_diffs:
    counts[d['type']] += 1

print('Difference classification (full text):')
for k in ['orthographic', 'variant', 'only_butcher', 'only_perseus']:
    print(f'  {k:20s}: {counts[k]:4d}')

Difference classification (full text):
  orthographic        :   61
  variant             :  137
  only_butcher        :   36
  only_perseus        :   44


In [126]:
# Show all classified differences in a scrollable table
html = '<h3>All differences (classified)</h3>'
html += '<div style="max-height:600px; overflow-y:auto">'
html += '<table border="1" cellpadding="4" style="border-collapse:collapse; font-size:13px">'
html += '<tr><th>#</th><th>Type</th><th>Butcher</th><th>Perseus</th></tr>'

type_colors = {
    'orthographic': '#e8f4fd',
    'variant': '#fff3cd',
    'only_butcher': '#ffd6d6',
    'only_perseus': '#d6ffd6',
}

for i, d in enumerate(all_diffs, 1):
    bg = type_colors.get(d['type'], '#fff')
    html += f'<tr style="background:{bg}">'
    html += f'<td>{i}</td><td>{d["type"]}</td>'
    html += f'<td>{d["butcher"]}</td><td>{d["perseus"]}</td>'
    html += '</tr>'

html += '</table></div>'
html += '<p><b>Key:</b> '
html += '🔵 orthographic (accent/breathing only) | '
html += '🟡 variant (different word/reading) | '
html += '🔴 only_butcher (extra in Butcher) | '
html += '🟢 only_perseus (missing from Butcher)</p>'
display(HTML(html))

#,Type,Butcher,Perseus
1,variant,Περὶ,περὶ
2,variant,ἢ,καὶ ἡ
3,variant,μετ᾿,μετ
4,variant,ἀνώνυμος τυγχάνει οὖσα,ἀνώνυμοι τυγχάνουσι
5,orthographic,ἐλεγειοποιούς,ἐλεγειοποιοὺς
6,only_butcher,τοῦτον,
7,variant,Ἐπεὶ,ἐπεὶ
8,only_butcher,τῇ,
9,only_perseus,,τῇ
10,variant,Ἔτι,ἔτι


## 10. Check for hyphen-rejoin failures

If the Butcher text still contains stray hyphens after normalization, those are failed rejoins — words that were incorrectly split.

In [127]:
# Look for any remaining hyphens in the normalized Butcher text
stray_hyphens = []
for ch in sorted(butcher.keys(), key=section_sort_key):
    for sec in sorted(butcher[ch].keys(), key=section_sort_key):
        text = butcher[ch][sec]
        # Find words containing a hyphen (not em-dash or similar)
        for match in re.finditer(r'\S*-\S*', text):
            word = match.group()
            # Skip legitimate dashes (em-dash, editorial marks)
            if word in ('—', '–') or word.startswith('—'):
                continue
            stray_hyphens.append((ch, sec, word, match.start()))

if stray_hyphens:
    print(f'⚠️  Found {len(stray_hyphens)} remaining hyphens in normalized Butcher text:')
    for ch, sec, word, pos in stray_hyphens[:30]:
        print(f'  Ch {ch}, §{sec}: "{word}"')
    if len(stray_hyphens) > 30:
        print(f'  ... and {len(stray_hyphens) - 30} more')
else:
    print('✅ No stray hyphens found — all line-break hyphens were successfully rejoined.')


✅ No stray hyphens found — all line-break hyphens were successfully rejoined.


## 11. Summary

Run this cell for a quick pass/fail overview.

In [128]:
print('=' * 60)
print('COMPARISON SUMMARY')
print('=' * 60)

# Chapter structure
b_chs = set(butcher.keys())
p_chs = set(perseus.keys())
if b_chs == p_chs:
    print(f'\n✅ Chapters: Both have {len(b_chs)} chapters (1-26)')
else:
    print(f'\n❌ Chapter mismatch!')
    if b_chs - p_chs:
        print(f'   Only in Butcher: {b_chs - p_chs}')
    if p_chs - b_chs:
        print(f'   Only in Perseus: {p_chs - b_chs}')

# Section structure
mismatched_chs = [r for r in rows if r['match'] == '❌']
if mismatched_chs:
    print(f'\n⚠️  Sections: {len(mismatched_chs)} chapters have different section counts')
    for r in mismatched_chs:
        print(f'   Ch {r["chapter"]}: Butcher={r["butcher_sections"]}, Perseus={r["perseus_sections"]}')
else:
    print('\n✅ Sections: Identical structure in all chapters')

# Word coverage
print(f'\n📊 Word-level alignment:')
print(f'   Butcher total words:  {len(b_all)}')
print(f'   Perseus total words:  {len(p_all)}')
print(f'   Overall similarity:   {ratio:.4f} ({ratio*100:.1f}%)')
print(f'   Only in Butcher:      {counts["only_butcher"]} groups')
print(f'   Only in Perseus:      {counts["only_perseus"]} groups')
print(f'   Orthographic diffs:   {counts["orthographic"]}')
print(f'   Textual variants:     {counts["variant"]}')

# Hyphen check
if stray_hyphens:
    print(f'\n⚠️  Stray hyphens: {len(stray_hyphens)} (possible rejoin failures)')
else:
    print(f'\n✅ Hyphen handling: All line-break hyphens successfully rejoined')

print()

COMPARISON SUMMARY

✅ Chapters: Both have 26 chapters (1-26)

✅ Sections: Identical structure in all chapters

📊 Word-level alignment:
   Butcher total words:  10252
   Perseus total words:  10256
   Overall similarity:   0.9688 (96.9%)
   Only in Butcher:      36 groups
   Only in Perseus:      44 groups
   Orthographic diffs:   61
   Textual variants:     137

✅ Hyphen handling: All line-break hyphens successfully rejoined

